<a href="https://colab.research.google.com/github/ale66/learn-datascience/blob/main/week-9/Ranking_world_cup/2026_FIFA_World_Cup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 2026 FIFA World Cup Simulation and Team Rating Analysis

This notebook simulates the 2026 FIFA World Cup using actual group data. It includes:
*   **Team Rating Systems:** Massey, Keener, and Markov.
*   **Simulation Phases:** Group stage advancement and direct elimination knockout rounds.
*   **Objective:** Predict match outcomes and determine a tournament winner.

### 1: Historical data on International Matches - Download + Read Data

The dataset `results.csv` is obtained from the 'International Football Results from 1872 to 2026' Kaggle dataset, which compiles international football match results.

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
HIST_DATA = "https://raw.githubusercontent.com/ale66/learn-datascience/main/week-9/Ranking_world_cup/results.csv"

In [ ]:
df = pd.read_csv(HIST_DATA)

df.head()

In [ ]:

df.tail(10)

Take historical data of the last 8 years, which contains two previous world cups

In [ ]:
df = df[df["date"] >= "2018-01-01"]

In [ ]:
print(df.shape)

df.head()

## 2026 FIFA World Cup - group stage

This section fetches the 2026 FIFA World Cup group stage data from a public JSON file. 

It then processes this data to extract the teams participating in each group, which will be used for the simulation.

In [ ]:
import requests

In [ ]:
URL = "https://raw.githubusercontent.com/openfootball/worldcup.json/master/2026/worldcup.json"


In [ ]:
data = requests.get(URL).json()

data


extract teams and their respective groups

In [ ]:
groups = {}

teams = set()

for match in data["matches"]:

    team1 = match.get("team1")
    team2 = match.get("team2")
    group = match.get("group")

    # skip knockout placeholders like W101 etc.
    if not team1 or not team2:
        continue

    # only group stage matches
    if group is None:
        continue

    if "Group" not in group:
        continue

    if group not in groups:
        groups[group] = set()

    groups[group].add(team1)
    groups[group].add(team2)

    teams.add(team1)
    teams.add(team2)


Prepare groups for visualisation.

In [ ]:
groups = {k: sorted(list(v)) for k, v in groups.items()}

teams = sorted(list(teams))

sanity check

In [ ]:
print("number of teams:", len(teams))
print("number of groups:", len(groups))

for g, t in groups.items():
    print(g, t)

Pick up the corresponding dataset

In [ ]:
df_48 = df[
    (df["home_team"].isin(teams)) &
    (df["away_team"].isin(teams))
].copy()

## 2: Massey Ratings (Linear Algebra Version)


Core Formula: $\overline{M}\mathbf{r}=\mathbf{p}$

where

*   $M$ = simple match matrix, always the same;
*   $\overline{M}$ = Massey's *mutilated* matrix;
*   $\mathbf{r}$ = ratings vector, to be solved, and
*   $\mathbf{p}$ = goal difference vector, known.

#### Construct the simple M matrix

1. Teams

In [ ]:
n = len(teams)

team_index = {t:i for i,t in enumerate(teams)}


2. Fill Matrix

In [ ]:
M = np.zeros((n, n))
p = np.zeros(n)

for _, row in df_48.iterrows():
    home = team_index[row["home_team"]]
    away = team_index[row["away_team"]]

    hs = row["home_score"]
    as_ = row["away_score"]

    if np.isnan(hs) or np.isnan(as_):
        continue

    diff = hs - as_

    # Massey matrix
    M[home, home] += 1
    M[away, away] += 1
    M[home, away] -= 1
    M[away, home] -= 1

    p[home] += diff
    p[away] -= diff

##### Now construct Massey's mutilated matrix $\overline{M}$

In [ ]:
M[-1, :] = 1
p[-1] = 0

1. Solve  $\overline{M}\mathbf{r}=\mathbf{p}$ by finding the least-squares solution, thanks to `numpy.linalg.lstsq`.

In [ ]:
ratings_massey = np.linalg.lstsq(M, p, rcond=None)[0]

5. Output

In [ ]:
massey_df = pd.DataFrame({
    "team": teams,
    "massey": ratings_massey
})

massey_df = massey_df.sort_values("massey", ascending=False)
massey_df.reset_index(drop=True, inplace=True)

massey_df.head(10)

## 3: Keener Ratings (Perron vector)

Core Idea: solve the eigenvalue problem 

$A\mathbf{r} = \lambda \mathbf{r}$ 

to find the dominant (Perron) vector, which will represent the team ratings.

If $A$ is irreducible and non-negative, the Perron-Frobenius theorem guarantees a unique positive eigenvector corresponding to the largest eigenvalue $\lambda$.


In [ ]:
A = np.zeros((n, n))


In [ ]:
# Filter out rows with NaN scores from df_48 for Keener ratings calculation
df_48_keener = df_48.dropna(subset=["home_score", "away_score"]).copy()

for _, row in df_48_keener.iterrows():
    i = team_index[row["home_team"]]
    j = team_index[row["away_team"]]

    A[i, j] += row["home_score"] + 1

    A[j, i] += row["away_score"] + 1


Normalisation

In [ ]:
col_sum = A.sum(axis=0)

# Avoid division by zero for teams that might not have played any match
# Replace 0 with 1 in col_sum where it's 0 to prevent NaN/Inf in A after division
col_sum[col_sum == 0] = 1

A = A / col_sum


##### Calculate Perron's eigenvector

In [ ]:
# Ensure A is not all zeros or contains NaNs
# Also handle cases where A might contain inf
if not np.all(A == 0) and not np.any(np.isnan(A)) and not np.any(np.isinf(A)):

    eigvals, eigvecs = np.linalg.eig(A)

    idx = np.argmax(eigvals.real)

    keener_ratings = eigvecs[:, idx].real

    # Key: Standardize (non-negative + normalize)
    keener_ratings = np.abs(keener_ratings)
    keener_ratings = keener_ratings / keener_ratings.sum()

    keener_df = pd.DataFrame({
        "team": teams,
        "rating": keener_ratings
    }).sort_values("rating", ascending=False)
    
    keener_df.reset_index(drop=True, inplace=True)

    print(keener_df.head(10))
else:
    print("Matrix A is invalid (all zeros, contains NaN, or Inf values). Cannot compute Keener ratings.")

## 4. Markov Ratings

Markov ratings model the strength of teams based on a Markov chain, where teams are states and the transitions between states are determined by match outcomes (goal scores). 

The core idea is to establish a transition matrix where $P_{ij}$ represents the 'strength' or 'likelihood' of team $i$ transitioning to or dominating team $j$. 

The stationary distribution of this Markov chain then provides the long-term ratings of each team.

### Construct the Markov Transition Matrix

In [ ]:
n = len(teams)


In [ ]:
A_markov = np.zeros((n, n))


In [ ]:
# Filter out matches with NaN scores
df_48_markov = df_48.dropna(subset=["home_score", "away_score"]).copy()

for _, row in df_48_markov.iterrows():

    home = row["home_team"]
    away = row["away_team"]

    i = team_index[home]
    j = team_index[away]

    hs = row["home_score"]
    aw = row["away_score"]

    # Home team wins
    if hs > aw:
        # Loser (away) gives weight to winner (home)
        A_markov[j, i] += 1

    # Away team wins
    elif aw > hs:
        A_markov[i, j] += 1

    # it's a tie
    else:
        A_markov[i, j] += 0.5
        A_markov[j, i] += 0.5
        

Handle teams with no outgoing edges (no matches played)

In [ ]:
for i in range(n):
    if A_markov[i].sum() == 0:
        A_markov[i] = np.ones(n)
        

Row normalization

Let's make the transition matrix stochastic: each row sums to 1.

In [ ]:
row_sums = A_markov.sum(axis=1)

S = A_markov / row_sums[:, None]

##### Damping/Escape factor

This is the escape probability, to be determined empirically. 

We begin with a lowish value of 0.15, which is PageRank, essentially. 

Experiment with higher values, up to 0.5, roughly. 

In [ ]:
ALPHA = 0.15


Create the Escape Matrix

In [ ]:
E = np.ones((n, n)) / n


In [ ]:
P_markov = (
    (1 - ALPHA) * S + ALPHA * E
)


Solve

In [ ]:
eigvals, eigvecs = np.linalg.eig(P_markov.T)

idx = np.argmin(np.abs(eigvals - 1))

ratings = np.real(eigvecs[:, idx])

ratings = ratings / ratings.sum()


Output rankings

In [ ]:
markov_df = pd.DataFrame({
    "team": teams,
    "rating": ratings
})

markov_df = markov_df.sort_values(
    "rating",
    ascending=False
)

markov_df.reset_index(drop=True, inplace=True)

print(markov_df.head(10))

### 5. Putting it All Together: 2026 World Cup Simulation

Create a single function `simulate_world_cup` that can be used with any rating system (Massey, Keener, Markov). This function will handle:

1.  **Group Stage Advancement:** Identifying the top two teams from each group and the best eight third-placed teams.
2.  **Knockout Rounds:** Simulating matches based on the provided ratings until a single winner is determined.

In [ ]:
import random

In [ ]:
def simulate_world_cup(ratings_df, rating_column_name, groups, simulation_name):
    """
    Simulates the World Cup tournament based on given team ratings.

    Args:
        ratings_df (pd.DataFrame): DataFrame containing 'team' and the specified rating_column_name.
        rating_column_name (str): The name of the column in ratings_df that holds the ratings.
        groups (dict): A dictionary where keys are group names and values are lists of team names.
        simulation_name (str): A string identifier for the current simulation (e.g., 'Massey', 'Keener').

    Returns:
        str: The name of the winning team.
    """
    print(f"\n--- World Cup Simulation ({simulation_name} Ratings) ---")

    advancing_teams = {}
    third_placed_teams = []

    # --- Group Stage Advancement ---
    print(f"\n### Group Stage Advancement ({simulation_name} Ratings)")
    for group_name, team_list in groups.items():
        group_teams_ratings = ratings_df[ratings_df['team'].isin(team_list)].copy()
        group_teams_ratings = group_teams_ratings.sort_values(by=rating_column_name, ascending=False)

        # Top two teams from each group automatically advance
        advancing_teams[group_name] = group_teams_ratings.head(2)['team'].tolist()

        # Store third-placed teams for potential advancement
        if len(group_teams_ratings) >= 3:
            third_placed_teams.append({
                'team': group_teams_ratings.iloc[2]['team'],
                'rating': group_teams_ratings.iloc[2][rating_column_name],
                'group': group_name
            })

    # Sort third-placed teams by rating and take the top 8
    third_placed_teams_df = pd.DataFrame(third_placed_teams)
    if not third_placed_teams_df.empty:
        third_placed_teams_df = third_placed_teams_df.sort_values(by='rating', ascending=False).head(8)

    print(f"Teams advancing as 1st and 2nd in groups ({simulation_name}):")
    for group, teams_adv in advancing_teams.items():
       print(f"{group}: {teams_adv}")

    print(f"\nTeams advancing as best 3rd placed ({simulation_name}):")
    if not third_placed_teams_df.empty:
        print(third_placed_teams_df[['group', 'team', 'rating']].to_string(index=False))
    else:
        print("No third-placed teams to advance.")


    # --- Knockout Stage Simulation ---
    print(f"\n### Knockout Stage Simulation ({simulation_name} Ratings)")

    def simulate_match(team1_name, team2_name, ratings_df, rating_col):
        rating1 = ratings_df[ratings_df['team'] == team1_name][rating_col].iloc[0]
        rating2 = ratings_df[ratings_df['team'] == team2_name][rating_col].iloc[0]

        if rating1 > rating2:
            return team1_name
        elif rating2 > rating1:
            return team2_name
        else:
            # If ratings are equal, pick a random winner
            return random.choice([team1_name, team2_name])

    # Consolidate all advancing teams into a single list for Round of 32
    round_of_32_teams = []

    for group_teams in advancing_teams.values():
        round_of_32_teams.extend(group_teams)

    if not third_placed_teams_df.empty:
        round_of_32_teams.extend(third_placed_teams_df['team'].tolist())

    print(f"\nTotal teams in Round of 32 ({simulation_name}): {len(round_of_32_teams)}")
    print(f"Teams in Round of 32 ({simulation_name}):")
    print(round_of_32_teams)

    # Shuffle teams to create initial pairings (for a simplified bracket)
    random.shuffle(round_of_32_teams)

    current_round_teams = round_of_32_teams
    
    round_number = 32

    while len(current_round_teams) > 1:
        print(f"\n--- Round of {round_number} ({simulation_name}) ---")
        winners = []

        # Ensure even number of teams for pairings, if odd, one team gets a bye
        if len(current_round_teams) % 2 != 0:
            print(f"Warning: Odd number of teams ({len(current_round_teams)}). One team will get a bye.")
            winners.append(current_round_teams.pop(0)) # Give a bye to the first team

        for i in range(0, len(current_round_teams), 2):
            team1 = current_round_teams[i]
            team2 = current_round_teams[i+1]

            winner = simulate_match(team1, team2, ratings_df, rating_column_name)
            print(f"{team1} vs {team2} -> Winner: {winner}")
            winners.append(winner)

        current_round_teams = winners
        round_number //= 2

    print(f"\n--- World Cup Winner ({simulation_name}) ---")
    print(current_round_teams[0])
    return current_round_teams[0]

### Run Simulations with the General Function

In [ ]:
massey_winner = simulate_world_cup(massey_df, 'massey', groups, 'Massey')

In [ ]:
keener_winner = simulate_world_cup(keener_df, 'rating', groups, 'Keener')

In [ ]:
markov_winner = simulate_world_cup(markov_df, 'rating', groups, 'Markov')

### Comparative Summary of Top Teams by Rating Method

In [ ]:
comparative_top_teams = pd.DataFrame({
    'Massey Ratings': massey_df.head(5)['team'].tolist(),
    'Keener Ratings': keener_df.head(5)['team'].tolist(),
    'Markov Ratings': markov_df.head(5)['team'].tolist()
})

print(f"Massey Winner: {massey_winner}")
print(f"Keener Winner: {keener_winner}")
print(f"Markov Winner: {markov_winner}")

display(comparative_top_teams)